In [74]:
import polars as pl

In [75]:
# df = pl.read
from pathlib import Path
p = Path("/home/ubuntu/fedmat/outputs/2025-12-09/05-58-31/train.log")
import re
pat = re.compile(r".*round ([0-9]+): accuracy=([0-9]+\.[0-9]+)")
# pat = re.compile(r".*accuracy=.*")
client_pat = re.compile(r".*Round ([0-9]+) client ([0-9]+).*avg_loss=([0-9]+\.[0-9]+)")

server_data = {
    "r": [],
    "acc": [],
}
client_data = {
    "r": [],
    "idx": [],
    "loss": [],
}
with p.open("r") as f:
    # l = f.readline()
    for l in f:
        if "Server eval" in l and (m:=pat.match(l)):
            # print(m.group())
            server_data["r"].append(m.group(1))
            server_data["acc"].append(m.group(2))
            print(l)
        elif "Round" in l and (m:=client_pat.match(l)):
            client_data["r"].append(m.group(1))
            client_data["idx"].append(m.group(2))
            client_data["loss"].append(m.group(3))
        if "New best" in l:
            print(l)

[rank=main world=5 backend=thread] [2025-12-09 05:59:05,355][fedmat.train][INFO] - Server eval after round 1: accuracy=0.1863

[rank=main world=5 backend=thread] [2025-12-09 05:59:05,517][fedmat.train][INFO] - New best server model (acc=0.1863) saved to /home/ubuntu/fedmat/outputs/2025-12-09/05-58-31/best-server.pt

[rank=main world=5 backend=thread] [2025-12-09 05:59:37,427][fedmat.train][INFO] - Server eval after round 2: accuracy=0.1921

[rank=main world=5 backend=thread] [2025-12-09 05:59:37,629][fedmat.train][INFO] - New best server model (acc=0.1921) saved to /home/ubuntu/fedmat/outputs/2025-12-09/05-58-31/best-server.pt

[rank=main world=5 backend=thread] [2025-12-09 06:00:08,433][fedmat.train][INFO] - Server eval after round 3: accuracy=0.1051

[rank=main world=5 backend=thread] [2025-12-09 06:00:39,105][fedmat.train][INFO] - Server eval after round 4: accuracy=0.1586

[rank=main world=5 backend=thread] [2025-12-09 06:01:09,789][fedmat.train][INFO] - Server eval after round 5: 

In [76]:
df = pl.DataFrame(server_data).cast({"r": pl.UInt64, "acc": pl.Float32})
cdf = pl.DataFrame(client_data).cast({"r": pl.UInt64, "idx": pl.UInt64, "loss": pl.Float32})

In [77]:
df[-1]

r,acc
u64,f32
147,0.4924


In [78]:
df["acc"].max()

0.5022000074386597

In [79]:
import plotly.express as px
px.scatter(x=df["r"], y=df["acc"], template="plotly_dark").show()
px.line(x=cdf["r"], y=cdf["loss"], color=cdf["idx"], template="plotly_dark")